# 🚀 Chapter 6: Advanced Logistic Regression and Extensions
**Referensi Buku:** *scikit-learn Cookbook, Third Edition*

---
## 1. Pendahuluan
Regresi Logistik bukan sekadar algoritma klasifikasi biner dasar. Dalam praktiknya, kita sering menghadapi dataset dengan banyak kategori (*multiclass*), dataset berukuran masif yang membutuhkan algoritma pengoptimalan (*solver*) khusus, atau data yang sangat tidak seimbang (*imbalanced*). Bab ini membahas fungsi-fungsi lanjutan dari `LogisticRegression` di scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

%matplotlib inline
np.random.seed(42)

## 2. Klasifikasi Multiclass (OVR vs Multinomial)
Secara default, Regresi Logistik memprediksi 2 kelas (Biner). Untuk lebih dari 2 kelas, scikit-learn menggunakan dua strategi:
- **One-Versus-Rest (OvR):** Membangun satu model biner untuk setiap kelas yang melawan semua kelas sisanya.
- **Multinomial (Softmax):** Mengoptimalkan fungsi kerugian (*loss*) secara langsung untuk seluruh kelas sekaligus. Biasanya lebih akurat tetapi lebih berat komputasinya.

In [ ]:
# Membuat data dengan 3 kelas (Multiclass)
X_multi, y_multi = make_blobs(n_samples=300, centers=3, n_features=2, random_state=42, cluster_std=2.0)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.3)

# Model 1: One-Versus-Rest (OvR)
log_ovr = LogisticRegression(multi_class='ovr', solver='lbfgs')
log_ovr.fit(X_train_m, y_train_m)

# Model 2: Multinomial (Cross-Entropy)
log_multi = LogisticRegression(multi_class='multinomial', solver='lbfgs')
log_multi.fit(X_train_m, y_train_m)

print(f"Akurasi OvR: {log_ovr.score(X_test_m, y_test_m):.3f}")
print(f"Akurasi Multinomial: {log_multi.score(X_test_m, y_test_m):.3f}")

## 3. Menyesuaikan Solver dan Regularisasi (SAGA & Elastic-Net)
Parameter `solver` sangat penting. Solver standar `lbfgs` hanya mendukung penalti L2 (Ridge). Jika dataset Anda sangat besar atau Anda ingin menggunakan penalti campuran L1 & L2 (Elastic-Net), Anda harus menggunakan solver **'saga'**.

In [ ]:
# Menggunakan solver 'saga' yang mendukung penalti 'elasticnet'
log_saga = LogisticRegression(solver='saga', penalty='elasticnet', l1_ratio=0.5, max_iter=5000)
log_saga.fit(X_train_m, y_train_m)

print("Koefisien Model (SAGA - ElasticNet):")
print(log_saga.coef_)
print(f"\nAkurasi SAGA ElasticNet: {log_saga.score(X_test_m, y_test_m):.3f}")

## 4. LogisticRegressionCV (Validasi Silang Bawaan)
Sama seperti `RidgeCV` atau `LassoCV`, scikit-learn memiliki kelas `LogisticRegressionCV` yang secara efisien mencari parameter regularisasi (`C` yang merupakan kebalikan dari `alpha`) secara otomatis tanpa perlu menggunakan `GridSearchCV` yang lebih lambat.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

# Mencari hyperparameter C secara otomatis dengan 5-fold cross validation
log_cv = LogisticRegressionCV(cv=5, random_state=42, max_iter=1000)
log_cv.fit(X_train_m, y_train_m)

print("Nilai C (Inverse Regularization Strength) terbaik yang ditemukan untuk tiap kelas:")
print(log_cv.C_)
print(f"\nAkurasi LogisticRegressionCV: {log_cv.score(X_test_m, y_test_m):.3f}")

## 5. Menangani Imbalanced Data dengan `class_weight`
Di dunia nyata (seperti deteksi penipuan/fraud), satu kelas (misal: fraud) mungkin hanya 1% dari total data. Model standar akan selalu memprediksi "Bukan Fraud" dan mendapatkan akurasi 99%, tetapi itu sama sekali tidak berguna.

Parameter `class_weight='balanced'` memaksa algoritma untuk memberikan denda (penalti) yang lebih besar saat membuat kesalahan pada kelas minoritas.

In [ ]:
# Membuat dataset sangat tidak seimbang (95% Kelas 0, 5% Kelas 1)
X_imb, y_imb = make_classification(n_samples=1000, n_classes=2, weights=[0.95, 0.05], 
                                   random_state=42, n_clusters_per_class=1)
Xi_train, Xi_test, yi_train, yi_test = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42)

# 1. Model Standar (Bias terhadap mayoritas)
log_standard = LogisticRegression()
log_standard.fit(Xi_train, yi_train)

# 2. Model Balanced (Memberi perhatian khusus pada minoritas)
log_balanced = LogisticRegression(class_weight='balanced')
log_balanced.fit(Xi_train, yi_train)

print("=== Perbandingan Recall (Kemampuan Mendeteksi Kelas Minoritas/Kelas 1) ===\n")
print("1. Logistic Regression Standar:")
print(classification_report(yi_test, log_standard.predict(Xi_test)))

print("\n2. Logistic Regression Balanced:")
print(classification_report(yi_test, log_balanced.predict(Xi_test)))

print("*(Perhatikan nilai Recall untuk kelas 1 pada model Balanced meningkat drastis dibandingkan model Standar)*")